<a href="https://colab.research.google.com/github/DeliaRudy/Storytelling/blob/multilingual-marketing-demo-12165702612363080727/business_story_telling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multilingual Marketing with Google Cloud AI -  Presented By Ruvimbo Delia Hakata**

Welcome to this lab! Today, we're going to demonstrate how you can leverage Google Cloud's powerful AI ecosystem to create a multilingual marketing campaign in just 5 minutes.

We'll be focusing on South African languages, specifically **English (South African)**, **Afrikaans**, **isiZulu**, and **isiXhosa**.

### **Our Workflow:**
1.  **Input:** User describes their ad copy or script.
2.  **Thinking by Gemini (Vertex AI):** Use Gemini to refine and polish the marketing script.
3.  **Google TTS API:** Generate high-quality audio for our generated script.
4.  **Translation (Vertex AI & Cloud Translation API):** Compare different ways to translate our content.
5.  **Multilingual Audio (Beyond standard APIs):** Show how to use Model Garden for even more language support.

## **0. Setup & Authentication**

First, we need to install the necessary libraries and authenticate with Google Cloud. In a Colab environment, this is usually handled by providing a service account key or using the built-in authentication.

In [ ]:
# Install necessary Google Cloud and helper libraries
# !pip install google-cloud-texttospeech google-cloud-translate google-cloud-aiplatform pydub

import os
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import texttospeech
from google.cloud import translate_v2 as translate
from google.cloud import aiplatform
from IPython.display import Audio, display

# @title Authentication Setup
# Replace with your actual GCP Project ID and Location
PROJECT_ID = "demotts-492017" # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)

# In a real lab, the user would provide their own credentials or use the Colab environment's auth.
# Commenting out the credentials path to use Colab's default authentication.
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/credentials.json"

## **1. Step 1: Input - Describe Your Ad**

Every great campaign starts with an idea. Let's capture the core message the user wants to convey.

In [ ]:
# @title Ad Script Input
# Ask the user to describe their ad copy or what the ad is about.
user_input = input("Describe your ad (e.g., 'A short 15s radio ad for a new luxury red lipstick called Rouge'): ")
print(f"\nInput received: {user_input}")

## **2. Step 2: Thinking by Gemini (Vertex AI)**

Now, we'll use Gemini Pro on Vertex AI to take that simple description and turn it into a professional, catchy 15-second radio script. This demonstrates how LLMs can act as creative partners.

In [ ]:
# @title Script Generation with Gemini

# Configure the Gemini Pro model from Vertex AI
model = GenerativeModel("gemini-1.5-pro-002")

# Build a prompt to guide Gemini
prompt = f"""
You are a professional copywriter. Create a short, punchy 15-second radio ad script based on this description: '{user_input}'.
The script should be in English and suitable for a South African audience.
Format the output as plain text, including only the spoken lines.
"""

# Call the Gemini model
response = model.generate_content(prompt)
english_script = response.text

print("--- Generated English Script ---")
print(english_script)

## **3. Step 3: Google TTS API (English)**

Let's hear our script! We'll use the Google Cloud Text-to-Speech API to generate a high-quality, natural-sounding voice with a South African accent.

In [ ]:
# @title Synthesizing English (South African) Audio

def synthesize_text(text, language_code, voice_name, output_filename):
    client = texttospeech.TextToSpeechClient()
    synthesis_input = texttospeech.SynthesisInput(text=text)

    # Select the voice parameters
    voice = texttospeech.VoiceSelectionParams(
        language_code=language_code,
        name=voice_name
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    with open(output_filename, "wb") as out:
        out.write(response.audio_content)

    print(f"Audio content written to file '{output_filename}'")
    display(Audio(output_filename))

# Generate English (South African) Audio
# Note: 'en-ZA-Wavenet-A' is a popular choice for South African English
synthesize_text(english_script, "en-ZA", "en-ZA-Wavenet-A", "ad_english.mp3")

## **4. Step 4: Translation - API vs. Vertex AI Model**

Now for the multilingual part! We'll show two ways to translate our script into **Afrikaans**, **isiZulu**, and **isiXhosa**.

### **Option A: Google Cloud Translation API**
A fast, reliable, and straightforward API for high-volume translations.

In [ ]:
# @title Translation using the Cloud Translation API

translate_client = translate.Client()

languages = {
    'af': 'Afrikaans',
    'zu': 'Zulu',
    'xh': 'Xhosa'
}

api_translations = {}
for lang_code, lang_name in languages.items():
    result = translate_client.translate(english_script, target_language=lang_code)
    api_translations[lang_code] = result['translatedText']
    print(f"--- {lang_name} (API) ---")
    print(result['translatedText'])
    print()

### **Option B: Translation using Vertex AI (Gemini)**
Using an LLM like Gemini allows for more context-aware and creative translations, often better for marketing copy.

In [ ]:
# @title Translation using Gemini (Vertex AI)

llm_translations = {}
for lang_code, lang_name in languages.items():
    prompt = f"Translate the following marketing script into {lang_name}. Keep the tone professional yet catchy for a South African audience: '{english_script}'"
    response = model.generate_content(prompt)
    llm_translations[lang_code] = response.text
    print(f"--- {lang_name} (Gemini) ---")
    print(response.text)
    print()

## **5. Step 5: New Copies of Audio**

Finally, we'll generate the audio for our translated scripts. For Afrikaans, we can use the standard TTS API. For Zulu and Xhosa, we'll demonstrate how you can leverage **Model Garden** on Vertex AI to use specialized models when standard APIs might not support your target language yet.

### **Afrikaans Audio (Cloud TTS)**

In [ ]:
# @title Synthesizing Afrikaans Audio
# Using the standard Cloud TTS API for Afrikaans
synthesize_text(llm_translations['af'], "af-ZA", "af-ZA-Standard-A", "ad_afrikaans.mp3")

### **Zulu & Xhosa Audio (Leveraging Model Garden)**

While the standard TTS API is constantly expanding, Vertex AI's **Model Garden** gives you access to a wide range of models from Google and the open-source community.

For example, you can deploy models like **SeamlessM4T** (from Meta, available in Model Garden) which supports over 100 languages for speech-to-speech and text-to-speech, including Zulu and Xhosa!

This demonstrates that with Google Cloud, you aren't limited by standard APIs – you can leverage any model that fits your needs.

In [ ]:
# @title Demo: Using a Model Garden Model for Zulu/Xhosa

def call_model_garden_tts(text, target_lang, output_filename):
    """
    This is a placeholder function to demonstrate how you would call
    a model deployed from Vertex AI Model Garden (e.g., SeamlessM4T).
    """
    print(f"--- Calling Model Garden Model for {target_lang} ---")
    # In a real scenario, you would deploy the model to an endpoint first:
    # endpoint = aiplatform.Endpoint("projects/.../locations/.../endpoints/...")
    # response = endpoint.predict(instances=[{"text": text, "target_lang": target_lang}])

    # This powerful capability allows you to bring specialized translation and TTS models
    # for languages like Zulu and Xhosa directly into your GCP environment.
    print(f"[Simulated] Sending text to Model Garden endpoint for {target_lang}...")
    print(f"✅ Successfully used a specialized model from Model Garden for {target_lang}!")

    # (In a real demo, we might have pre-generated files or show the endpoint calling logic)
    # display(Audio(f"pre_generated_{target_lang}.mp3"))

# Demonstrate for Zulu and Xhosa
call_model_garden_tts(llm_translations['zu'], "Zulu", "ad_zulu.mp3")
call_model_garden_tts(llm_translations['xh'], "Xhosa", "ad_xhosa.mp3")

## **Summary**

In just 5 minutes, we have:
1.  **Refined** a marketing idea using **Gemini on Vertex AI**.
2.  **Generated** natural-sounding audio in local accents.
3.  **Translated** that script into multiple local languages using both **Cloud Translation API** and **Vertex AI**.
4.  **Demonstrated** how to go beyond standard APIs by leveraging **Model Garden** for broader language support.

This shows the incredible power and flexibility of Google Cloud AI for reaching every customer in their preferred language!